# Import Libraries

In [4]:
%pip install pandas regex==2022.9.13 torch gensim tqdm ipywidgets scikit-learn scipy

In [5]:
import os
import sys
from pathlib import Path

import re
import torch
from collections import Counter
import pandas as pd
import numpy as np
import gensim.downloader as api
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

sys.path.append(os.getcwd())

from dataset import *
from model import *
from trainer import Trainer


torch.manual_seed(42)

# Read Data

In [6]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train.head()

,rate,text
0,4,Очень понравилось. Были в начале марта с соба...
1,5,В целом магазин устраивает.\nАссортимент позво...
2,5,"Очень хорошо что открылась 5 ка, теперь не над..."
3,3,Пятёрочка громко объявила о том как она заботи...
4,3,"Тесно, вечная сутолока, между рядами трудно ра..."


# Label encoding

In [7]:
def advanced_clean_text(text):
    """Многоуровневая очистка текста"""
    if not isinstance(text, str):
        return ""

    text = text.lower()

    text = re.sub(r'[^\w\sа-яё]', ' ', text)

    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    text = re.sub(r'\d+', 'число', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

train['text_cleaned'] = train['text'].apply(advanced_clean_text)
test['text_cleaned'] = test['text'].apply(advanced_clean_text)
train['rate'] = train['rate'] - 1

# Create pretrained tokenizers

In [8]:
import re
from collections import Counter
from itertools import chain
from typing import List, Optional
import numpy as np

import torch
from torch.nn.utils.rnn import pack_sequence, pad_sequence
from torch.utils.data import Dataset, DataLoader


class Tokenizer:
    """Продвинутый токенизатор с поддержкой стоп-слов и стемминга"""

    def __init__(self, use_stopwords=True, use_stemming=False):
        self.word_pattern = re.compile(r'[а-яёa-z]+(?:-[а-яёa-z]+)?')
        self.use_stopwords = use_stopwords
        self.use_stemming = use_stemming

        # Русские стоп-слова (можно расширить)
        self.stop_words = {
            'и', 'в', 'во', 'не', 'что', 'на', 'я', 'с', 'со', 'как', 'а',
            'но', 'он', 'по', 'у', 'же', 'из', 'за', 'то', 'так', 'же',
            'вы', 'весь', 'это', 'все', 'о', 'при', 'к', 'очень', 'было',
            'только', 'еще', 'уже', 'вот', 'бы', 'да', 'нет', 'для', 'до',
            'или', 'без', 'через', 'над', 'под', 'об', 'про', 'этот', 'тот'
        }

        # Простой стеммер для русского языка
        self.suffixes = {
            'ую', 'юю', 'ая', 'яя', 'ое', 'ее', 'ые', 'ие', 'ой', 'ей',
            'ый', 'ий', 'ого', 'его', 'ому', 'ему', 'ым', 'им', 'ом', 'ем',
            'ать', 'ять', 'ить', 'еть', 'уть', 'ют', 'ит', 'ет', 'ат', 'ят'
        }

    def tokenize(self, text: str) -> List[str]:
        """Токенизация с очисткой"""
        if not isinstance(text, str):
            return []

        # Приводим к нижнему регистру
        text = text.lower()
        # Находим все слова
        tokens = self.word_pattern.findall(text)

        # Удаляем короткие слова (меньше 2 букв)
        tokens = [t for t in tokens if len(t) >= 2]

        # Удаляем стоп-слова
        if self.use_stopwords:
            tokens = [t for t in tokens if t not in self.stop_words]

        # Стемминг (упрощённый)
        if self.use_stemming:
            tokens = [self._stem(t) for t in tokens]

        return tokens

    def _stem(self, word: str) -> str:
        """Простой суффиксный стеммер"""
        for suffix in self.suffixes:
            if word.endswith(suffix):
                word = word[:-len(suffix)]
                break
        return word


class Vocab:
    def __init__(self, tokenized_texts: List[List[str]], max_vocab_size=None, min_freq=2):
        """
        Словарь с поддержкой минимальной частоты и предобученных эмбеддингов
        """
        # Подсчёт частот
        counts = Counter(chain(*tokenized_texts))

        # Фильтрация по минимальной частоте
        if min_freq > 1:
            counts = {word: cnt for word, cnt in counts.items() if cnt >= min_freq}

        max_vocab_size = max_vocab_size or len(counts)

        # Берём самые частотные слова
        common_pairs = Counter(counts).most_common(max_vocab_size)

        # Специальные токены
        self.PAD_IDX = 0
        self.UNK_IDX = 1
        self.EOS_IDX = 2
        self.SOS_IDX = 3  # Начало предложения (для encoder-decoder)

        self.itos = ["<PAD>", "<UNK>", "<EOS>", "<SOS>"] + [pair[0] for pair in common_pairs]
        self.stoi = {token: i for i, token in enumerate(self.itos)}

        # Словарь для быстрого поиска частот
        self.word_freq = counts

    def vectorize(self, text: List[str], add_sos=False):
        """
        Векторизация с опциональным добавлением токена начала
        """
        tokens = [self.stoi.get(tok, self.UNK_IDX) for tok in text]
        if add_sos:
            tokens = [self.SOS_IDX] + tokens
        tokens.append(self.EOS_IDX)
        return tokens

    def __len__(self):
        return len(self.itos)


class TextDataset(Dataset):
    def __init__(self, tokenized_texts, labels, vocab, max_seq_len=None):
        """
        Датасет с паддингом до максимальной длины
        """
        self.texts = tokenized_texts
        self.labels = labels
        self.vocab = vocab
        self.max_seq_len = max_seq_len or self._get_max_len(tokenized_texts)

    def _get_max_len(self, texts):
        """Вычисление 95-го перцентиля длин текстов"""
        lengths = [len(t) for t in texts]
        return int(np.percentile(lengths, 95))

    def __getitem__(self, item):
        vectorized = self.vocab.vectorize(self.texts[item])

        # Обрезаем слишком длинные тексты
        if len(vectorized) > self.max_seq_len:
            vectorized = vectorized[:self.max_seq_len - 1] + [self.vocab.EOS_IDX]

        return (
            torch.tensor(vectorized, dtype=torch.long),
            torch.tensor(self.labels[item], dtype=torch.long)
        )

    def __len__(self):
        return len(self.texts)

    def collate_fn(self, batch):
        """Кастомная функция для паддинга"""
        texts, labels = zip(*batch)

        # Паддинг до максимальной длины в батче
        padded_texts = pad_sequence(texts, batch_first=True, padding_value=self.vocab.PAD_IDX)
        labels = torch.tensor(labels, dtype=torch.long)

        # Длины последовательностей для pack_padded_sequence
        lengths = torch.tensor([len(t) for t in texts], dtype=torch.long)

        return padded_texts, labels, lengths


def train_test_split(data, train_frac=0.85, random_state=42):
    """
    Стратифицированное разделение с перемешиванием
    """
    np.random.seed(random_state)
    n_toxicity_ratings = 5
    train_labels = []
    val_labels = []
    train_texts = []
    val_texts = []

    for label in range(n_toxicity_ratings):
        texts = data[data.rate == label].text.values
        # Перемешиваем
        indices = np.random.permutation(len(texts))
        texts = texts[indices]

        n_train = int(len(texts) * train_frac)
        train_texts.extend(texts[:n_train])
        val_texts.extend(texts[n_train:])
        train_labels += [label] * n_train
        val_labels += [label] * (len(texts) - n_train)

    return train_texts, train_labels, val_texts, val_labels

In [9]:
tokenizer = Tokenizer(use_stopwords=True, use_stemming=False)
tokenized_texts = [tokenizer.tokenize(text) for text in train['text_cleaned']]

In [10]:
lengths = [len(t) for t in tokenized_texts]
print(f"\nСтатистика длин текстов:")
print(f"  Средняя: {np.mean(lengths):.1f}")
print(f"  Медиана: {np.median(lengths):.1f}")
print(f"  95-й перцентиль: {np.percentile(lengths, 95):.1f}")
print(f"  Максимум: {max(lengths)}")


Статистика длин текстов:
  Средняя: 13.8
  Медиана: 10.0
  95-й перцентиль: 39.0
  Максимум: 443


# Splitting Data

In [11]:
MAX_VOCAB_SIZE = 50000
MIN_WORD_FREQ = 2
vocab = Vocab(tokenized_texts, max_vocab_size=MAX_VOCAB_SIZE, min_freq=MIN_WORD_FREQ)
print(f"\nРазмер словаря: {len(vocab)}")

train_texts, train_labels, val_texts, val_labels = train_test_split(train, train_frac=0.85)

# Создаём датасеты с фиксированной максимальной длиной
MAX_SEQ_LEN = 256  # Обрезаем до 256 токенов

train_dataset = TextDataset(
    [tokenizer.tokenize(text) for text in train_texts],
    train_labels,
    vocab,
    max_seq_len=MAX_SEQ_LEN
)

val_dataset = TextDataset(
    [tokenizer.tokenize(text) for text in val_texts],
    val_labels,
    vocab,
    max_seq_len=MAX_SEQ_LEN
)



Размер словаря: 20923


In [12]:
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to('cpu')

# Create embeddings from tokens

In [13]:
os.environ["GENSIM_DATA_DIR"] = str(Path.cwd())
gensim_model = api.load("word2vec-ruscorpora-300")
emb_matrix = prepare_emb_matrix(gensim_model, vocab, use_pretrained=True)

[==================================================] 100.0% 198.8/198.8MB downloaded


# Init Model and Config

In [14]:
config = {
    # Архитектура
    "freeze": False,                    # Дообучаем эмбеддинги
    "cell_type": "LSTM",                # LSTM > GRU для длинных текстов
    "cell_dropout": 0.3,                # Dropout между слоями LSTM
    "num_layers": 3,                    # Глубина
    "hidden_size": 256,                 # Размер скрытого состояния
    "bidirectional": True,              # Двунаправленный
    "use_attention": True,              # Механизм внимания
    "emb_dropout": 0.2,                 # Dropout на эмбеддингах
    "out_dropout": 0.4,                 # Dropout в классификаторе
    "out_sizes": [128, 64],             # Промежуточные слои
}


trainer_config = {
    "lr": 0.001,                        # Learning rate
    "n_epochs": 30,                     # Максимум эпох (ранняя остановка)
    "weight_decay": 1e-4,               # L2 регуляризация
    "batch_size": 64,                   # Размер батча
    "patience": 7,                      # Эпох без улучшения до остановки
    "device": 'cpu',
    "verbose": True
}

# Create Dataloaders and Train

In [15]:
# Взвешенное сэмплирование для балансировки классов
class_dist = {}
for label in train_labels:
    class_dist[label] = class_dist.get(label, 0) + 1

print(f"Распределение классов в train: {class_dist}")

# Вычисляем веса для сэмплирования (обратная частота)
sample_weights = [1.0 / class_dist[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=trainer_config["batch_size"],
    sampler=sampler,
    num_workers=2,
    collate_fn=train_dataset.collate_fn,
    pin_memory=False
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=trainer_config["batch_size"],
    shuffle=False,
    num_workers=2,
    collate_fn=val_dataset.collate_fn,
    pin_memory=False
)

Распределение классов в train: {0: 3517, 1: 2048, 2: 5207, 3: 8433, 4: 22158}


In [16]:
model = RecurrentClassifier(config, vocab, emb_matrix)
trainer = Trainer(trainer_config)
trainer.fit(model, train_dataloader, val_dataloader)


Эпоха 1/30
LR: 0.001000


Обучение:   0%|          | 0/647 [00:00<?, ?it/s]

Валидация:   0%|          | 0/115 [00:00<?, ?it/s]

Train Loss: 1.3803, Train Acc: 0.4402
Val Loss: 1.1806, Val Acc: 0.5644, Val F1: 0.5839
✓ Новая лучшая модель! F1: 0.5839

Эпоха 2/30
LR: 0.001000


Обучение:   0%|          | 0/647 [00:00<?, ?it/s]

...

Train Loss: 0.6659, Train Acc: 0.8787
Val Loss: 1.3520, Val Acc: 0.5935, Val F1: 0.5971

Эпоха 13/30
LR: 0.000309


Обучение:   0%|          | 0/647 [00:00<?, ?it/s]

Валидация:   0%|          | 0/115 [00:00<?, ?it/s]

Train Loss: 0.6386, Train Acc: 0.8917
Val Loss: 1.3824, Val Acc: 0.5839, Val F1: 0.5918

Ранняя остановка на эпохе 13. Лучший F1: 0.6099

Лучший F1 на валидации: 0.6099


RecurrentClassifier(
  (embeddings): Embedding(20923, 300, padding_idx=0)
  (emb_dropout): Dropout(p=0.2, inplace=False)
  (cell): LSTM(300, 256, num_layers=3, batch_first=True, dropout=0.3, bidirectional=True)
  (attention): Attention(
    (attention): Linear(in_features=512, out_features=1, bias=True)
  )
  (out_dropout): Dropout(p=0.4, inplace=False)
  (out_proj): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.4, inplace=False)
    (8): Linear(in_features=64, out_features=5, bias=True)
  )
)

# Define predict function

In [17]:
val_predictions = trainer.predict(val_dataloader)

Предсказание:   0%|          | 0/115 [00:00<?, ?it/s]

In [18]:
print(f"F1-score (weighted): {f1_score(val_labels, val_predictions, average='weighted'):.4f}")
print(f"F1-score (macro): {f1_score(val_labels, val_predictions, average='macro'):.4f}")
print("\nClassification Report:")
print(classification_report(val_labels, val_predictions))

F1-score (weighted): 0.6099
F1-score (macro): 0.4584

Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.52      0.50       621
           1       0.18      0.18      0.18       362
           2       0.36      0.45      0.40       919
           3       0.39      0.44      0.42      1489
           4       0.84      0.74      0.79      3911

    accuracy                           0.60      7302
   macro avg       0.45      0.47      0.46      7302
weighted avg       0.63      0.60      0.61      7302



In [19]:
cm = confusion_matrix(val_labels, val_predictions)
print("\nConfusion Matrix (истинные классы 0-4):")
print(cm)


Confusion Matrix (истинные классы 0-4):
[[ 320   94  133   40   34]
 [ 101   66  130   44   21]
 [ 116  100  412  219   72]
 [  54   60  285  661  429]
 [  57   42  190  711 2911]]


# Get testset predictions

In [20]:
test_dataset = TextDataset(
    [tokenizer.tokenize(text) for text in test['text_cleaned'].values],
    [-1] * len(test),
    vocab,
    max_seq_len=MAX_SEQ_LEN
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=trainer_config["batch_size"],
    shuffle=False,
    num_workers=2,
    collate_fn=test_dataset.collate_fn,
    pin_memory=False
)

test_predictions = trainer.predict(test_dataloader)
test_predictions = test_predictions + 1  # Возвращаем к 1-5

Предсказание:   0%|          | 0/191 [00:00<?, ?it/s]

# Create submission

In [21]:
sample_submission = pd.read_csv("sample_submission.csv")
sample_submission["rate"] = test_predictions
sample_submission.head()

,index,rate
0,0,5
1,1,2
2,2,5
3,3,3
4,4,3


In [23]:
sample_submission.to_csv("submission_rnn_final.csv", index=False)